# Exp4 Baseline: Sequential vs PRNG-Interleaved Reconstruction

**Purpose**

This notebook reproduces the retrospective Exp4 Baseline reconstruction discussed on 2026-08-12.

It does three things:

1. Reconstructs the original Exp4 sequential Subject/PCS streams.
2. Reconstructs a locally interleaved version using one fixed PRNG rule within each adjacent bit pair.
3. Tests the physical first-half vs second-half H_RS difference directly, before Subject/PCS labeling, including the early-vs-late session-position check.

**Important interpretation**

The interleaved reconstruction is a retrospective architecture diagnostic on the same frozen QRNG calls. It is not an independent dataset and it is not a prospective interleaved experiment.

The physical-half test asks whether the first 150 physical data bits differ from the second 150 physical data bits in H_RS. It does not use the Subject/PCS labels.


In [1]:
import os, ast, pickle, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

# Mount Drive (safe to call repeatedly).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped/failed (may already be mounted):", e)

# Recursive search from the current working directory (matches the pattern
# used by the other exp5-prescreen notebooks) so it finds each file no
# matter which Drive subfolder it's actually sitting in, instead of
# depending on one hardcoded path. Wildcards tolerate a download suffix
# such as "(1)".
REQUIRED_INPUTS = {
    "blocks": "Frozen_Blocks_2026-02-10_195735.csv",
    "sessions": "Frozen_Sessions_2026-02-10_195735.csv",
    "raw_calls": "Frozen_Exp4_RawBlockBits_2026-07-26*.pkl",
}

def find_file(pattern, search_root=None):
    root = Path.cwd() if search_root is None else Path(search_root)
    candidates = []
    for parent in (root, root / "data"):
        if parent.is_dir():
            candidates.extend(p.resolve() for p in parent.glob(pattern) if p.is_file())
    if not candidates:
        candidates = [p.resolve() for p in root.rglob(pattern) if p.is_file()]
    candidates = sorted(set(candidates))
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find {pattern}. Upload it to Colab, add it to Drive, "
            f"or change search_root. Current search root: {root.resolve()}"
        )
    if len(candidates) > 1:
        locations = ", ".join(str(p) for p in candidates)
        raise FileNotFoundError(f"Multiple copies of {pattern} found: {locations}")
    return str(candidates[0])

BLOCKS = find_file(REQUIRED_INPUTS["blocks"])
SESSIONS = find_file(REQUIRED_INPUTS["sessions"])
RAW = find_file(REQUIRED_INPUTS["raw_calls"])

print(BLOCKS)
print(SESSIONS)
print(RAW)


Mounted at /content/drive
/content/drive/MyDrive/QART_Project/Frozen_Blocks_2026-02-10_195735.csv
/content/drive/MyDrive/QART_Project/Frozen_Sessions_2026-02-10_195735.csv
/content/drive/MyDrive/Frozen_Exp4_RawBlockBits_2026-07-26.pkl


In [2]:
# Exact Python translation of the production JavaScript hurstApprox logic.
def hurst_approx(bits):
    n = len(bits)
    if n < 20:
        return 0.5

    x = np.array([1 if int(b) else -1 for b in bits], dtype=float)
    mean = x.mean()
    d = x - mean
    y = np.cumsum(d)

    # JS initializes minY=maxY=0 before accumulating.
    R = max(0.0, float(y.max())) - min(0.0, float(y.min()))
    S = float(np.sqrt(np.mean(d * d))) or 1.0

    value = np.log((R / S) if (R / S) else 1.0) / np.log(n)
    return max(0.0, min(1.0, float(value)))


In [3]:
blocks = pd.read_csv(BLOCKS)
sessions = pd.read_csv(SESSIONS)
with open(RAW, "rb") as f:
    raw = pickle.load(f)

baseline_ids = set(
    sessions.loc[sessions["agent_class"].eq("baseline"), "sessionId"]
)
base = blocks[blocks["sessionId"].isin(baseline_ids)].copy()

print(f"Baseline blocks: {len(base):,}")
print(f"Baseline sessions: {base['sessionId'].nunique()}")


Baseline blocks: 3,090
Baseline sessions: 103


## 1. Reconstruct original and PRNG-interleaved streams

For each 301-bit call:

- bit 0 remains the original assignment bit
- bits 1-300 are arranged into 150 adjacent pairs
- a fixed PRNG rule chooses which member of each pair goes to stream A
- the other member goes to stream B
- the original bit 0 then labels A/B as Subject/PCS

The fixed global seed below makes the reconstruction reproducible.


In [4]:
GLOBAL_SEED = "2026-08-12-exp4-baseline-interleave-v1"

records = []

for _, row in base.sort_values(["sessionId", "block_idx"]).iterrows():
    sid = row["sessionId"]
    idx = int(row["block_idx"])

    raw_map = dict(raw[sid])
    call = raw_map[idx]

    assignment_bit = int(call[0])
    data_bits = list(map(int, call[1:301]))

    # Original physical halves.
    halfA = data_bits[:150]
    halfB = data_bits[150:]

    # Reproducible per-block PRNG seed derived from the one fixed global seed.
    seed_material = f"{GLOBAL_SEED}|{sid}|{idx}".encode()
    seed_int = int.from_bytes(
        hashlib.sha256(seed_material).digest()[:8], "big"
    )
    rng = np.random.default_rng(seed_int)

    # Locally balanced interleaving:
    # one bit from every adjacent pair enters A, the other enters B.
    choices = rng.integers(0, 2, size=150)
    A, B = [], []

    for j, c in enumerate(choices):
        first = data_bits[2*j]
        second = data_bits[2*j + 1]
        if c == 0:
            A.append(first)
            B.append(second)
        else:
            A.append(second)
            B.append(first)

    inter_subject = A if assignment_bit == 1 else B
    inter_pcs = B if assignment_bit == 1 else A

    # Original saved Subject/PCS bits.
    td = ast.literal_eval(row["trial_data"])
    original_subject = td["subject_bits"]
    original_pcs = td["demon_bits"]

    records.append({
        "sessionId": sid,
        "block_idx": idx,
        "assignment_bit": assignment_bit,

        "HRS_subject_original": hurst_approx(original_subject),
        "HRS_pcs_original": hurst_approx(original_pcs),
        "delta_HRS_original":
            hurst_approx(original_subject) - hurst_approx(original_pcs),

        "HRS_subject_interleaved": hurst_approx(inter_subject),
        "HRS_pcs_interleaved": hurst_approx(inter_pcs),
        "delta_HRS_interleaved":
            hurst_approx(inter_subject) - hurst_approx(inter_pcs),

        # Physical halves BEFORE Subject/PCS labeling.
        "HRS_halfA": hurst_approx(halfA),
        "HRS_halfB": hurst_approx(halfB),
        "delta_halfA_minus_halfB":
            hurst_approx(halfA) - hurst_approx(halfB),

        "stream_A_interleaved": "".join(map(str, A)),
        "stream_B_interleaved": "".join(map(str, B)),
        "subject_interleaved": "".join(map(str, inter_subject)),
        "pcs_interleaved": "".join(map(str, inter_pcs)),
    })

df = pd.DataFrame(records)
df.head()


,sessionId,block_idx,assignment_bit,HRS_subject_original,HRS_pcs_original,delta_HRS_original,HRS_subject_interleaved,HRS_pcs_interleaved,delta_HRS_interleaved,HRS_halfA,HRS_halfB,delta_halfA_minus_halfB,stream_A_interleaved,stream_B_interleaved,subject_interleaved,pcs_interleaved
0,10eO7RvBJxIeBrY2o6jk,0,1,0.565881,0.549619,0.016262,0.586591,0.525707,0.060884,0.565881,0.549619,0.016262,0111100001010000100011110000110001110110100001...,1011011010000000110000100110001100011011111100...,0111100001010000100011110000110001110110100001...,1011011010000000110000100110001100011011111100...
1,10eO7RvBJxIeBrY2o6jk,1,0,0.480186,0.547782,-0.067595,0.596432,0.569333,0.027098,0.547782,0.480186,0.067595,1111111011000000101110101001111000000001000010...,1000001011100000011101000011011111110000001110...,1000001011100000011101000011011111110000001110...,1111111011000000101110101001111000000001000010...
2,10eO7RvBJxIeBrY2o6jk,2,1,0.487797,0.607503,-0.119706,0.576847,0.540354,0.036493,0.487797,0.607503,-0.119706,1100101000001110001000011110001010110111000010...,0011000101101011001100111110101010010110010100...,1100101000001110001000011110001010110111000010...,0011000101101011001100111110101010010110010100...
3,10eO7RvBJxIeBrY2o6jk,3,1,0.594815,0.491739,0.103076,0.521547,0.482650,0.038897,0.594815,0.491739,0.103076,0000000100011011011011111111011000101001000011...,1100001101111111001001100110011100101000010101...,0000000100011011011011111111011000101001000011...,1100001101111111001001100110011100101000010101...
4,10eO7RvBJxIeBrY2o6jk,4,1,0.513347,0.507744,0.005603,0.538226,0.565100,-0.026874,0.513347,0.507744,0.005603,0100101010000101101010010110101111010101001111...,1001011001110000101001010011000000110101010111...,0100101010000101101010010110101111010101001111...,1001011001110000101001010011000000110101010111...


## 2. Original vs interleaved Baseline summary

In [5]:
def early_late_contrast(data, column):
    early = data.loc[data["block_idx"] <= 14, column].mean()
    late = data.loc[data["block_idx"] >= 15, column].mean()
    return early, late, early - late

orig_e, orig_l, orig_diff = early_late_contrast(df, "delta_HRS_original")
int_e, int_l, int_diff = early_late_contrast(df, "delta_HRS_interleaved")

summary = pd.DataFrame({
    "measure": [
        "Mean delta_HRS",
        "Median delta_HRS",
        "Block SD delta_HRS",
        "Proportion above 0",
        "Proportion below 0",
        "Early mean delta_HRS",
        "Late mean delta_HRS",
        "Early minus late",
    ],
    "original_sequential": [
        df["delta_HRS_original"].mean(),
        df["delta_HRS_original"].median(),
        df["delta_HRS_original"].std(ddof=1),
        (df["delta_HRS_original"] > 0).mean(),
        (df["delta_HRS_original"] < 0).mean(),
        orig_e, orig_l, orig_diff,
    ],
    "prng_interleaved": [
        df["delta_HRS_interleaved"].mean(),
        df["delta_HRS_interleaved"].median(),
        df["delta_HRS_interleaved"].std(ddof=1),
        (df["delta_HRS_interleaved"] > 0).mean(),
        (df["delta_HRS_interleaved"] < 0).mean(),
        int_e, int_l, int_diff,
    ]
})

summary


,measure,original_sequential,prng_interleaved
0,Mean delta_HRS,-0.001741,-0.000962
1,Median delta_HRS,-0.000579,0.000093
2,Block SD delta_HRS,0.064154,0.064922
3,Proportion above 0,0.495793,0.500647
4,Proportion below 0,0.503883,0.498706
5,Early mean delta_HRS,-0.004065,-0.000281
6,Late mean delta_HRS,0.000583,-0.001642
7,Early minus late,-0.004647,0.001362


## 3. Direct physical-half test

This is the mechanism check.

For each original call, before applying the assignment bit:

- **Half A** = physical bits 1-150
- **Half B** = physical bits 151-300
- calculate `H_RS(A) - H_RS(B)`

Then compare that physical-half difference in early blocks (0-14) versus late blocks (15-29).

If the previously observed Subject/PCS early-vs-late result were simply a direct manifestation of a first-half/second-half H_RS asymmetry, we would expect a corresponding resolved early-vs-late contrast here.


In [6]:
phys_overall = df["delta_halfA_minus_halfB"].mean()
phys_sd = df["delta_halfA_minus_halfB"].std(ddof=1)
phys_early, phys_late, phys_diff = early_late_contrast(
    df, "delta_halfA_minus_halfB"
)

physical_summary = pd.DataFrame({
    "measure": [
        "Overall mean A minus B",
        "Block SD A minus B",
        "Early mean A minus B",
        "Late mean A minus B",
        "Early minus late"
    ],
    "value": [
        phys_overall,
        phys_sd,
        phys_early,
        phys_late,
        phys_diff
    ]
})

physical_summary


,measure,value
0,Overall mean A minus B,-0.000682
1,Block SD A minus B,0.064174
2,Early mean A minus B,0.000640
3,Late mean A minus B,-0.002004
4,Early minus late,0.002644


## 4. Session-level bootstrap uncertainty

Baseline inference is resampled at the whole-session level rather than treating blocks as independent.

The bootstrap below uses 20,000 resamples and reports percentile 95% intervals.


In [7]:
BOOT_SEED = 20260812
N_BOOT = 20_000

rng = np.random.default_rng(BOOT_SEED)
session_ids = np.array(sorted(df["sessionId"].unique()))
by_session = {sid: df[df["sessionId"] == sid] for sid in session_ids}

boot_overall = np.empty(N_BOOT)
boot_early_late = np.empty(N_BOOT)

for b in range(N_BOOT):
    sampled = rng.choice(session_ids, size=len(session_ids), replace=True)
    tmp = pd.concat([by_session[sid] for sid in sampled], ignore_index=True)

    boot_overall[b] = tmp["delta_halfA_minus_halfB"].mean()

    early = tmp.loc[
        tmp["block_idx"] <= 14, "delta_halfA_minus_halfB"
    ].mean()
    late = tmp.loc[
        tmp["block_idx"] >= 15, "delta_halfA_minus_halfB"
    ].mean()
    boot_early_late[b] = early - late

results = pd.DataFrame({
    "estimand": [
        "Physical Half A minus Half B overall",
        "Physical-half early minus late"
    ],
    "point_estimate": [
        phys_overall,
        phys_diff
    ],
    "CI_2.5%": [
        np.quantile(boot_overall, .025),
        np.quantile(boot_early_late, .025)
    ],
    "CI_97.5%": [
        np.quantile(boot_overall, .975),
        np.quantile(boot_early_late, .975)
    ]
})

results


,estimand,point_estimate,CI_2.5%,CI_97.5%
0,Physical Half A minus Half B overall,-0.000682,-0.00306,0.001687
1,Physical-half early minus late,0.002644,-0.00156,0.006955


## 5. Save reconstruction outputs

In [8]:
df.to_csv("Exp4_Baseline_PRNG_Interleaved_Reconstruction.csv", index=False)
summary.to_csv("Exp4_Baseline_PRNG_Interleaved_Summary.csv", index=False)
physical_summary.to_csv("Exp4_Baseline_PhysicalHalf_Summary.csv", index=False)
results.to_csv("Exp4_Baseline_PhysicalHalf_Bootstrap.csv", index=False)

print("Saved:")
print("  Exp4_Baseline_PRNG_Interleaved_Reconstruction.csv")
print("  Exp4_Baseline_PRNG_Interleaved_Summary.csv")
print("  Exp4_Baseline_PhysicalHalf_Summary.csv")
print("  Exp4_Baseline_PhysicalHalf_Bootstrap.csv")


Saved:
  Exp4_Baseline_PRNG_Interleaved_Reconstruction.csv
  Exp4_Baseline_PRNG_Interleaved_Summary.csv
  Exp4_Baseline_PhysicalHalf_Summary.csv
  Exp4_Baseline_PhysicalHalf_Bootstrap.csv


## 6. Reconstruction-null test: could the PRNG itself explain the apparent improvement?

This test asks whether the specific fixed PRNG seed used above happened to produce an unusually favorable interleaving.

We repeat the adjacent-pair interleaving reconstruction across 500 independent fixed seeds. For every reconstruction:

- each adjacent pair contributes one bit to A and one bit to B
- the PRNG decides which member of the pair goes to A
- the original Exp4 assignment bit labels A/B as Subject/PCS
- the same H_RS calculation is applied
- the overall Baseline mean and early-minus-late contrast are recorded

We also compute a deterministic odd/even interleaving with no PRNG selection at all. This provides a direct check of whether the disappearance of the early/late effect depends on using a PRNG.


In [9]:
# Vectorized helpers for repeated reconstructions
raw_rows = []
for _, row in base.sort_values(["sessionId", "block_idx"]).iterrows():
    sid = row["sessionId"]
    idx = int(row["block_idx"])
    call = dict(raw[sid])[idx]
    assignment = int(call[0])
    data_bits = list(map(int, call[1:301]))
    td = ast.literal_eval(row["trial_data"])
    original_delta = (
        hurst_approx(td["subject_bits"]) - hurst_approx(td["demon_bits"])
    )
    raw_rows.append((sid, idx, assignment, data_bits, original_delta))

sids = np.array([r[0] for r in raw_rows], dtype=object)
idxs = np.array([r[1] for r in raw_rows], dtype=int)
assign = np.array([r[2] for r in raw_rows], dtype=int)
data = np.array([r[3] for r in raw_rows], dtype=np.int8)
orig_delta = np.array([r[4] for r in raw_rows], dtype=float)
pairs = data.reshape(len(raw_rows), 150, 2)

def hurst_matrix(bits01):
    x = np.where(bits01 == 1, 1.0, -1.0)
    mean = x.mean(axis=1, keepdims=True)
    d = x - mean
    y = np.cumsum(d, axis=1)
    ymin = np.minimum(0.0, y.min(axis=1))
    ymax = np.maximum(0.0, y.max(axis=1))
    R = ymax - ymin
    S = np.sqrt(np.mean(d*d, axis=1))
    S = np.where(S == 0, 1.0, S)
    ratio = np.where((R/S) > 0, R/S, 1.0)
    return np.clip(np.log(ratio) / np.log(bits01.shape[1]), 0, 1)

early_mask = idxs <= 14
late_mask = idxs >= 15

def interleave_for_seed(seed):
    rng = np.random.default_rng(seed)
    choices = rng.integers(0, 2, size=(len(raw_rows), 150))
    rr = np.arange(len(raw_rows))[:, None]
    cc = np.arange(150)[None, :]
    A = pairs[rr, cc, choices]
    B = pairs[rr, cc, 1 - choices]

    hA = hurst_matrix(A)
    hB = hurst_matrix(B)
    return np.where(assign == 1, hA - hB, hB - hA)

N_SEEDS = 500
seed_rows = []

for seed in range(N_SEEDS):
    d = interleave_for_seed(seed)
    seed_rows.append({
        "seed": seed,
        "mean_delta_HRS": d.mean(),
        "sd_delta_HRS": d.std(ddof=1),
        "early_minus_late": (
            d[early_mask].mean() - d[late_mask].mean()
        )
    })

seed_results = pd.DataFrame(seed_rows)

original_early_late = (
    orig_delta[early_mask].mean() - orig_delta[late_mask].mean()
)

seed_summary = pd.DataFrame({
    "quantity": [
        "Interleaved mean delta_HRS",
        "Interleaved early-minus-late",
        "Interleaved block SD"
    ],
    "2.5%": [
        seed_results["mean_delta_HRS"].quantile(.025),
        seed_results["early_minus_late"].quantile(.025),
        seed_results["sd_delta_HRS"].quantile(.025),
    ],
    "median": [
        seed_results["mean_delta_HRS"].median(),
        seed_results["early_minus_late"].median(),
        seed_results["sd_delta_HRS"].median(),
    ],
    "97.5%": [
        seed_results["mean_delta_HRS"].quantile(.975),
        seed_results["early_minus_late"].quantile(.975),
        seed_results["sd_delta_HRS"].quantile(.975),
    ]
})

print("Original sequential early-minus-late:", original_early_late)
print()
display(seed_summary)

print("\nEmpirical one-sided tail area for original early-late contrast:",
      np.mean(seed_results["early_minus_late"] <= original_early_late))
print("Empirical two-sided tail area:",
      np.mean(np.abs(seed_results["early_minus_late"]) >= abs(original_early_late)))


Original sequential early-minus-late: -0.004647367498499477



,quantity,2.5%,median,97.5%
0,Interleaved mean delta_HRS,-0.002245,-0.000092,0.002085
1,Interleaved early-minus-late,-0.004166,-0.000163,0.004026
2,Interleaved block SD,0.062557,0.064046,0.065583



Empirical one-sided tail area for original early-late contrast: 0.016
Empirical two-sided tail area: 0.03


### Deterministic odd/even check

This version uses **no PRNG at all**:

- A receives the first member of every adjacent pair
- B receives the second member of every adjacent pair

If this also removes the original early/late asymmetry, then the effect cannot reasonably be attributed specifically to the PRNG selector.


In [10]:
A_odd = pairs[:, :, 0]
B_even = pairs[:, :, 1]

hA_odd = hurst_matrix(A_odd)
hB_even = hurst_matrix(B_even)

delta_odd_even = np.where(
    assign == 1,
    hA_odd - hB_even,
    hB_even - hA_odd
)

odd_even_summary = pd.DataFrame({
    "measure": [
        "Mean delta_HRS",
        "Block SD delta_HRS",
        "Early minus late"
    ],
    "value": [
        delta_odd_even.mean(),
        delta_odd_even.std(ddof=1),
        (
            delta_odd_even[early_mask].mean()
            - delta_odd_even[late_mask].mean()
        )
    ]
})

odd_even_summary


,measure,value
0,Mean delta_HRS,-0.000034
1,Block SD delta_HRS,0.063214
2,Early minus late,0.001418


In [11]:
seed_results.to_csv(
    "Exp4_Baseline_Interleaving_500Seed_Null.csv", index=False
)
seed_summary.to_csv(
    "Exp4_Baseline_Interleaving_500Seed_Summary.csv", index=False
)
odd_even_summary.to_csv(
    "Exp4_Baseline_OddEven_Interleaving_Summary.csv", index=False
)

print("Saved reconstruction-null outputs.")


Saved reconstruction-null outputs.


## 7. AI condition: sequential vs interleaved early/late check

This repeats the same architecture check in the AI-agent condition. AI is useful here because, like Baseline, it was automated, while Human sessions had participant pacing.

The test compares the original sequential Subject−PCS ΔH_RS with a fixed-PRNG adjacent-pair interleaved reconstruction and bootstraps whole AI sessions.


In [12]:
# AI analysis uses the same reconstruction logic as above.
ai_ids = set(sessions.loc[sessions["agent_class"].eq("ai_agent"), "sessionId"])
ai = blocks[blocks["sessionId"].isin(ai_ids)].copy()

GLOBAL_SEED_AI = "2026-08-12-exp4-ai-interleave-v1"
ai_rows = []

for _, row in ai.sort_values(["sessionId", "block_idx"]).iterrows():
    sid = row["sessionId"]
    idx = int(row["block_idx"])
    call = dict(raw[sid])[idx]
    assignment = int(call[0])
    data_bits = list(map(int, call[1:301]))
    td = ast.literal_eval(row["trial_data"])

    seed_material = f"{GLOBAL_SEED_AI}|{sid}|{idx}".encode()
    seed_int = int.from_bytes(hashlib.sha256(seed_material).digest()[:8], "big")
    rng = np.random.default_rng(seed_int)
    choices = rng.integers(0, 2, size=150)

    A, B = [], []
    for j, c in enumerate(choices):
        first, second = data_bits[2*j], data_bits[2*j+1]
        if c == 0:
            A.append(first); B.append(second)
        else:
            A.append(second); B.append(first)

    inter_subject = A if assignment == 1 else B
    inter_pcs = B if assignment == 1 else A

    ai_rows.append({
        "sessionId": sid,
        "block_idx": idx,
        "orig_delta": hurst_approx(td["subject_bits"]) - hurst_approx(td["demon_bits"]),
        "int_delta": hurst_approx(inter_subject) - hurst_approx(inter_pcs)
    })

ai_df = pd.DataFrame(ai_rows)

def summarize_ai(col):
    early = ai_df.loc[ai_df.block_idx <= 14, col].mean()
    late = ai_df.loc[ai_df.block_idx >= 15, col].mean()
    return ai_df[col].mean(), ai_df[col].std(ddof=1), early, late, early-late

ai_orig = summarize_ai("orig_delta")
ai_inter = summarize_ai("int_delta")

ai_session_ids = np.array(sorted(ai_df.sessionId.unique()))
ai_by_session = {sid: ai_df[ai_df.sessionId == sid] for sid in ai_session_ids}
rng = np.random.default_rng(20260812)
N_BOOT_AI = 20_000
bo = np.empty(N_BOOT_AI)
bi = np.empty(N_BOOT_AI)

for b in range(N_BOOT_AI):
    sampled = rng.choice(ai_session_ids, size=len(ai_session_ids), replace=True)
    tmp = pd.concat([ai_by_session[s] for s in sampled], ignore_index=True)
    bo[b] = (tmp.loc[tmp.block_idx <= 14, "orig_delta"].mean()
             - tmp.loc[tmp.block_idx >= 15, "orig_delta"].mean())
    bi[b] = (tmp.loc[tmp.block_idx <= 14, "int_delta"].mean()
             - tmp.loc[tmp.block_idx >= 15, "int_delta"].mean())

ai_summary = pd.DataFrame({
    "construction": ["Original sequential", "PRNG interleaved"],
    "mean_delta_HRS": [ai_orig[0], ai_inter[0]],
    "block_SD": [ai_orig[1], ai_inter[1]],
    "early_mean": [ai_orig[2], ai_inter[2]],
    "late_mean": [ai_orig[3], ai_inter[3]],
    "early_minus_late": [ai_orig[4], ai_inter[4]],
    "CI_low": [np.quantile(bo,.025), np.quantile(bi,.025)],
    "CI_high": [np.quantile(bo,.975), np.quantile(bi,.975)]
})
ai_summary


,construction,mean_delta_HRS,block_SD,early_mean,late_mean,early_minus_late,CI_low,CI_high
0,Original sequential,0.001185,0.063900,0.002438,-0.000081,0.002519,-0.001463,0.006334
1,PRNG interleaved,-0.000836,0.064491,-0.000537,-0.001138,0.000601,-0.003537,0.004727


In [13]:
ai_summary.to_csv("Exp4_AI_Sequential_vs_Interleaved_EarlyLate.csv", index=False)
print("Saved Exp4_AI_Sequential_vs_Interleaved_EarlyLate.csv")


Saved Exp4_AI_Sequential_vs_Interleaved_EarlyLate.csv


## 8. Baseline stream-length variance scan

This exploratory scan asks whether longer interleaved H_RS streams reduce block-level paired variability.

**Important limitation:** Exp4 only contains 300 experimental data bits per block. To evaluate stream lengths above 150 bits, this section concatenates successive Baseline block bit sequences **within session boundaries** and then forms locally interleaved A/B streams from adjacent pairs. This is useful for estimating the finite-sample behavior of the implemented H_RS score, but it is **not equivalent to having collected one longer 600-, 1200-, or 2400-bit QRNG API call**. It crosses original block boundaries and therefore cannot establish the exact behavior of a future longer-call implementation.

The deterministic odd/even interleave is used as the primary scan so that changing PRNG draws do not add another source of variability. A paired-PRNG interleave is included as sensitivity.


In [14]:
# Stream-length scan.
lengths = [50,75,100,150,200,250,300,400,500,600,750,900,1200]

# Build each Baseline session's continuous sequence from the 300 data bits
# in each retained block, preserving within-session block order.
session_raw = {}
for sid, g in base.groupby("sessionId"):
    bits = []
    for _, row in g.sort_values("block_idx").iterrows():
        call = dict(raw[sid])[int(row.block_idx)]
        bits.extend(map(int, call[1:301]))
    session_raw[sid] = bits

scan_rows = []
scan_rng = np.random.default_rng(20260812)

for L in lengths:
    deltas_det = []
    deltas_prng = []

    for sid, bits in session_raw.items():
        n_chunks = len(bits) // (2*L)

        for c in range(n_chunks):
            chunk = np.array(
                bits[c*2*L:(c+1)*2*L], dtype=np.int8
            ).reshape(L,2)

            # Deterministic local interleaving.
            A = chunk[:,0].tolist()
            B = chunk[:,1].tolist()
            deltas_det.append(hurst_approx(A)-hurst_approx(B))

            # Sensitivity: one PRNG choice within each adjacent pair.
            choice = scan_rng.integers(0,2,size=L)
            rr = np.arange(L)
            Ap = chunk[rr,choice].tolist()
            Bp = chunk[rr,1-choice].tolist()
            deltas_prng.append(hurst_approx(Ap)-hurst_approx(Bp))

    for method, vals in [
        ("odd_even", deltas_det),
        ("paired_PRNG", deltas_prng)
    ]:
        arr = np.asarray(vals)
        scan_rows.append({
            "stream_length": L,
            "method": method,
            "n_paired_streams": len(arr),
            "mean_delta_HRS": arr.mean(),
            "sd_delta_HRS": arr.std(ddof=1),
            "mean_abs_delta": np.abs(arr).mean()
        })

length_scan = pd.DataFrame(scan_rows)
length_scan


,stream_length,method,n_paired_streams,mean_delta_HRS,sd_delta_HRS,mean_abs_delta
0,50,odd_even,9270,0.000154,0.083480,0.066807
1,50,paired_PRNG,9270,-0.002125,0.083805,0.067224
2,75,odd_even,6180,0.000898,0.074369,0.059674
3,75,paired_PRNG,6180,0.000448,0.074375,0.059705
4,100,odd_even,4635,-0.000342,0.070026,0.056528
5,100,paired_PRNG,4635,0.000331,0.071164,0.057075
6,150,odd_even,3090,-0.001956,0.063184,0.051062
7,150,paired_PRNG,3090,0.000762,0.065122,0.052193
8,200,odd_even,2266,-0.000938,0.060792,0.048463
9,200,paired_PRNG,2266,-0.000099,0.059605,0.047577


In [15]:
# Key deterministic results.
key_lengths = [150,300,600,750,900,1200]
key = (
    length_scan[
        (length_scan.method=="odd_even")
        & (length_scan.stream_length.isin(key_lengths))
    ]
    .copy()
)

sd150 = float(
    key.loc[key.stream_length==150, "sd_delta_HRS"].iloc[0]
)
key["SD reduction vs 150 (%)"] = (
    1 - key["sd_delta_HRS"]/sd150
)*100

key


,stream_length,method,n_paired_streams,mean_delta_HRS,sd_delta_HRS,mean_abs_delta,SD reduction vs 150 (%)
6,150,odd_even,3090,-0.001956,0.063184,0.051062,0.000000
12,300,odd_even,1545,-0.001075,0.054495,0.043639,13.752487
18,600,odd_even,721,-0.000300,0.049024,0.039876,22.410811
20,750,odd_even,618,-0.000767,0.045911,0.036449,27.337372
22,900,odd_even,515,-0.001699,0.046499,0.036631,26.406875
24,1200,odd_even,309,-0.000315,0.044557,0.036004,29.481064


### Interpretation boundary

Longer streams reduce the observed block-level spread, but the reduction is gradual. The 300-bit-per-stream reconstruction reduces SD relative to 150 bits, but does not make individual paired H_RS values cluster tightly around zero.

A descriptive power-law fit may be used to summarize the observed trend, but extrapolation beyond the tested lengths is not a design guarantee because:

1. H_RS itself changes finite-sample behavior as `n` changes.
2. Long reconstructed streams cross original Exp4 block boundaries.
3. A future 600- or 1200-bit API call may have different temporal/provider behavior.
4. "Stable enough" requires a prespecified target, such as a desired block-level SD or desired uncertainty on an aggregated Baseline mean.


In [16]:
# Descriptive-only power-law fit to deterministic scan.
d = length_scan[length_scan.method=="odd_even"].sort_values("stream_length")
coef = np.polyfit(
    np.log(d["stream_length"]),
    np.log(d["sd_delta_HRS"]),
    1
)
b, loga = coef
a = np.exp(loga)

print(f"Descriptive fit: SD ≈ {a:.4f} × L^({b:.3f})")
for target in [0.05,0.045,0.04,0.035,0.03]:
    est_L = (target/a)**(1/b)
    print(f"Approx. L for SD {target:.3f}: {est_L:,.0f} bits/stream")

print("\nDo not use these extrapolated lengths as a preregistered design guarantee.")


Descriptive fit: SD ≈ 0.1730 × L^(-0.196)
Approx. L for SD 0.050: 564 bits/stream
Approx. L for SD 0.045: 966 bits/stream
Approx. L for SD 0.040: 1,763 bits/stream
Approx. L for SD 0.035: 3,485 bits/stream
Approx. L for SD 0.030: 7,654 bits/stream

Do not use these extrapolated lengths as a preregistered design guarantee.


In [17]:
length_scan.to_csv(
    "Exp4_Baseline_StreamLength_Variance_Scan.csv", index=False
)
key.to_csv(
    "Exp4_Baseline_StreamLength_KeyResults.csv", index=False
)
print("Saved stream-length scan outputs.")


Saved stream-length scan outputs.
